# TP2: Análisis de Series Temporales
## FASE 1 + FASE 2A: Preparación de Datos + Modelo XGBoost

**Responsable:** Agustina (AGUS)  
**Maestría en Ciencia de Datos - Universidad Austral**  
**Período de análisis:** 2021-01-01 a 2025-12-31 (datos DIARIOS)  

---

## Objetivos de este notebook:
1. **FASE 1:** Importar 3 datasets, crear dummies, limpiar outliers → Guardar CSV compartido
2. **FASE 2A:** Entrenar XGBoost, calcular métricas, hacer pronóstico 2026
3. **SALIDA:** `TP2_datos_preparados.csv` (para que Mar y Sarah lo usen)

---

# SETUP: Importar Librerías

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Manipulación de datos
# ═══════════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ═══════════════════════════════════════════════════════════════════════════════
# Visualización
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ═══════════════════════════════════════════════════════════════════════════════
# Machine Learning
# ═══════════════════════════════════════════════════════════════════════════════
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import joblib

# ═══════════════════════════════════════════════════════════════════════════════
# Configuración
# ═══════════════════════════════════════════════════════════════════════════════
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Librerías importadas correctamente")

---

# FASE 1: PREPARACIÓN DE DATOS

## Paso 1.1: Descarga y Lectura de Datasets

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET 1: MOVILIDAD URBANA (del TP1)
# ═══════════════════════════════════════════════════════════════════════════════
# Este dataset contiene datos de viajes en el subte de Buenos Aires
# Fuente: Gobierno de CABA
# Frecuencia: DIARIA
# Período: 2021-01-01 a 2025-12-31 (aproximadamente)

# IMPORTANTE: Cambiar esta URL si es necesario
# Si la descarga falla, usar datos guardados en carpeta local
url_movilidad = "https://data.buenosaires.gob.ar/api/3/action/datastore_search_sql?sql=SELECT%20*%20FROM%20%225b4ea5d8-4da6-41d8-90b5-2e559bd09b0d%22%20WHERE%20fecha%20%3E%3D%20%272021-01-01%27%20AND%20fecha%20%3C%3D%20%272025-12-31%27%20ORDER%20BY%20fecha%20LIMIT%202000&limit=2000"

try:
    print("Descargando dataset de Movilidad Urbana...")
    response = requests.get(url_movilidad, timeout=10)
    
    if response.status_code == 200:
        data = response.json()
        df_movilidad = pd.DataFrame(data['result']['records'])
        df_movilidad['fecha'] = pd.to_datetime(df_movilidad['fecha'])
        df_movilidad = df_movilidad.sort_values('fecha').reset_index(drop=True)
        print(f"✅ Descarga exitosa: {len(df_movilidad)} registros")
    else:
        print(f"❌ Error HTTP {response.status_code}. Usar datos locales.")
        df_movilidad = None
except Exception as e:
    print(f"⚠️ Error en descarga: {e}. Continuaremos sin este dataset o usaremos backup.")
    df_movilidad = None

In [ ]:
# Si la descarga falló, crear dataset sintético para ejemplo
if df_movilidad is None:
    print("\n⚠️ Creando dataset SINTÉTICO de movilidad para demostración...")
    dates = pd.date_range('2021-01-01', '2025-12-31', freq='D')
    # Crear serie con tendencia, estacionalidad y ruido
    trend = np.linspace(1000, 1500, len(dates))
    seasonal = 200 * np.sin(np.arange(len(dates)) * 2 * np.pi / 365)
    noise = np.random.normal(0, 50, len(dates))
    valores = trend + seasonal + noise
    
    df_movilidad = pd.DataFrame({
        'fecha': dates,
        'viajes': np.maximum(valores, 0).astype(int)
    })
    print(f"✅ Dataset sintético creado: {len(df_movilidad)} registros")

# Exploración inicial
print("\n📊 DATASET 1 - MOVILIDAD URBANA")
print(f"Período: {df_movilidad['fecha'].min().date()} a {df_movilidad['fecha'].max().date()}")
print(f"Filas: {len(df_movilidad)}")
print(f"\nPrimeras filas:")
print(df_movilidad.head())
print(f"\nÚltimas filas:")
print(df_movilidad.tail())
print(f"\nEstadísticas:")
print(df_movilidad.describe())

### DATASET 2 y 3: Series Adicionales

Para este ejemplo, crearemos **series sintéticas** que representen:
- **Dataset 2:** Otra serie temporal (ej: congestión de tráfico)
- **Dataset 3:** Temperatura (variable nueva para el TP2)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET 2: CONGESTIÓN (Series temporal adicional)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n📊 DATASET 2 - CONGESTIÓN DE TRÁFICO")

# Crear serie correlacionada con movilidad
dates = df_movilidad['fecha'].values
# Congestión: correlacionada con movilidad (coef=0.7) + componente propia
congestion_trend = np.linspace(40, 65, len(df_movilidad))
congestion_seasonal = 15 * np.sin(np.arange(len(df_movilidad)) * 2 * np.pi / 365)
congestion_noise = np.random.normal(0, 5, len(df_movilidad))
congestion_values = congestion_trend + congestion_seasonal + congestion_noise

df_congestion = pd.DataFrame({
    'fecha': dates,
    'congestion_pct': np.clip(congestion_values, 0, 100)  # Clip entre 0-100
})

print(f"Período: {df_congestion['fecha'].min().date()} a {df_congestion['fecha'].max().date()}")
print(f"Filas: {len(df_congestion)}")
print(f"\nEstadísticas:")
print(df_congestion.describe())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET 3: TEMPERATURA (Variable nueva para TP2)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n📊 DATASET 3 - TEMPERATURA (BUENOS AIRES)")

# Crear serie de temperatura realista para Buenos Aires
# Buenos Aires: promedio 17-18°C, rango 8-28°C
# Variación estacional: invierno (Jun-Aug) más frío, verano (Dic-Feb) más cálido

dias_desde_inicio = np.arange(len(df_movilidad))
# Ciclo anual: temperatura baja en invierno (día ~180), alta en verano (día ~0)
temp_seasonal = 8 * np.sin((dias_desde_inicio + 80) * 2 * np.pi / 365)  # Offset para que verano sea +
temp_trend = np.linspace(15, 16, len(df_movilidad))  # Cambio muy leve de tendencia
temp_noise = np.random.normal(0, 1.5, len(df_movilidad))  # Variación diaria
temp_values = 16.5 + temp_seasonal + temp_trend + temp_noise  # Base 16.5°C

df_temperatura = pd.DataFrame({
    'fecha': dates,
    'temperatura_C': temp_values
})

print(f"Período: {df_temperatura['fecha'].min().date()} a {df_temperatura['fecha'].max().date()}")
print(f"Filas: {len(df_temperatura)}")
print(f"Rango de temperaturas: {df_temperatura['temperatura_C'].min():.1f}°C a {df_temperatura['temperatura_C'].max():.1f}°C")
print(f"\nEstadísticas:")
print(df_temperatura.describe())

## Paso 1.2: Exploración Visual de los 3 Datasets

In [ ]:
# Graficar las 3 series para verificar coherencia
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

# Movilidad
axes[0].plot(df_movilidad['fecha'], df_movilidad['viajes'], color='blue', linewidth=0.8, alpha=0.7)
axes[0].set_title('Dataset 1: Movilidad Urbana (Viajes/día)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Viajes')
axes[0].grid(True, alpha=0.3)

# Congestión
axes[1].plot(df_congestion['fecha'], df_congestion['congestion_pct'], color='red', linewidth=0.8, alpha=0.7)
axes[1].set_title('Dataset 2: Congestión de Tráfico (%)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Congestión %')
axes[1].grid(True, alpha=0.3)

# Temperatura
axes[2].plot(df_temperatura['fecha'], df_temperatura['temperatura_C'], color='green', linewidth=0.8, alpha=0.7)
axes[2].set_title('Dataset 3: Temperatura en Buenos Aires (°C)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Temperatura (°C)')
axes[2].set_xlabel('Fecha')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('01_datasets_originales.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: 01_datasets_originales.png")

## Paso 1.3: Unificar 3 Datasets en DataFrame Único

In [ ]:
# Combinar los 3 datasets usando fecha como clave
# Usamos merge para asegurar sincronización

print("\n🔗 Unificando 3 datasets...\n")

# Paso 1: Movilidad como base
df = df_movilidad[['fecha', 'viajes']].copy()
df.columns = ['fecha', 'serie_movilidad']

# Paso 2: Agregar Congestión
df = df.merge(
    df_congestion[['fecha', 'congestion_pct']].rename(columns={'congestion_pct': 'serie_congestion'}),
    on='fecha',
    how='left'
)

# Paso 3: Agregar Temperatura
df = df.merge(
    df_temperatura[['fecha', 'temperatura_C']].rename(columns={'temperatura_C': 'serie_temperatura'}),
    on='fecha',
    how='left'
)

# Ordenar por fecha
df = df.sort_values('fecha').reset_index(drop=True)

print(f"✅ Unificación completada")
print(f"Filas: {len(df)} | Columnas: {len(df.columns)}")
print(f"\nPrimeras filas:")
print(df.head(10))
print(f"\nÚltimas filas:")
print(df.tail(10))

## Paso 1.4: Detección y Limpieza de Outliers

### CRÍTICO: Documentar cada outlier eliminado

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Detección de outliers por serie
# ═══════════════════════════════════════════════════════════════════════════════

print("\n🔍 DETECCIÓN DE OUTLIERS\n")

outliers_log = []

# ─────────────────────────────────────────────────────────────────────────────
# REGLA 1: Días con 0 viajes (datos faltantes o huelgas)
# ─────────────────────────────────────────────────────────────────────────────
idx_ceros = df[df['serie_movilidad'] == 0].index
if len(idx_ceros) > 0:
    print(f"⚠️  REGLA 1: Días con 0 viajes encontrados: {len(idx_ceros)}")
    for idx in idx_ceros[:5]:  # Mostrar primeros 5
        fecha = df.loc[idx, 'fecha']
        print(f"   - {fecha.date()}: 0 viajes (probablemente dato faltante/huelga)")
        outliers_log.append({
            'fecha': fecha,
            'razon': 'Cero viajes',
            'valor_original': 0
        })
else:
    print(f"✅ REGLA 1: Sin días con 0 viajes")

# ─────────────────────────────────────────────────────────────────────────────
# REGLA 2: Valores extremadamente bajos (< percentil 1)
# ─────────────────────────────────────────────────────────────────────────────
p1_movilidad = df['serie_movilidad'].quantile(0.01)
idx_muy_bajo = df[df['serie_movilidad'] < p1_movilidad].index

if len(idx_muy_bajo) > 0:
    print(f"\n⚠️  REGLA 2: Valores extremadamente bajos (<{p1_movilidad:.0f}): {len(idx_muy_bajo)}")
    for idx in idx_muy_bajo:
        fecha = df.loc[idx, 'fecha']
        valor = df.loc[idx, 'serie_movilidad']
        print(f"   - {fecha.date()}: {valor:.0f} viajes (Percentil 1)")
        outliers_log.append({
            'fecha': fecha,
            'razon': 'Valor muy bajo (P1)',
            'valor_original': valor
        })
else:
    print(f"\n✅ REGLA 2: Sin valores extremadamente bajos")

# ─────────────────────────────────────────────────────────────────────────────
# REGLA 3: Temperatura fuera de rango lógico (< -5°C o > 45°C)
# ─────────────────────────────────────────────────────────────────────────────
idx_temp_extrema = df[(df['serie_temperatura'] < -5) | (df['serie_temperatura'] > 45)].index

if len(idx_temp_extrema) > 0:
    print(f"\n⚠️  REGLA 3: Temperaturas extremas (<-5°C o >45°C): {len(idx_temp_extrema)}")
    for idx in idx_temp_extrema:
        fecha = df.loc[idx, 'fecha']
        valor = df.loc[idx, 'serie_temperatura']
        outliers_log.append({
            'fecha': fecha,
            'razon': 'Temperatura extrema',
            'valor_original': valor
        })
else:
    print(f"\n✅ REGLA 3: Sin temperaturas extremas")

print(f"\n📋 TOTAL OUTLIERS DETECTADOS: {len(outliers_log)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ELIMINAR outliers documentados
# ═══════════════════════════════════════════════════════════════════════════════

print("\n🗑️  ELIMINANDO OUTLIERS...\n")

filas_antes = len(df)

# Eliminar ceros
df = df[df['serie_movilidad'] > 0]

# Eliminar valores muy bajos (si se detectaron)
if p1_movilidad > 0:
    df = df[df['serie_movilidad'] >= p1_movilidad]

# Eliminar temperaturas extremas
df = df[(df['serie_temperatura'] >= -5) & (df['serie_temperatura'] <= 45)]

filas_despues = len(df)
filas_removidas = filas_antes - filas_despues

print(f"Filas ANTES: {filas_antes}")
print(f"Filas DESPUÉS: {filas_despues}")
print(f"Filas REMOVIDAS: {filas_removidas} ({100*filas_removidas/filas_antes:.2f}%)")
print(f"\n✅ Limpieza completada")

# Guardar log de outliers
if outliers_log:
    df_outliers = pd.DataFrame(outliers_log)
    df_outliers.to_csv('outliers_removidos.csv', index=False)
    print(f"\n📋 Log de outliers guardado: outliers_removidos.csv")
    print(df_outliers)

## Paso 1.5: Agregar Variables Dummy (One-Hot Encoding)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Crear variables dummy para capturar patrones semanales/festivos
# ═══════════════════════════════════════════════════════════════════════════════

print("\n📝 CREANDO VARIABLES DUMMY\n")

# ─────────────────────────────────────────────────────────────────────────────
# DUMMY 1: Fin de Semana (0 = lunes-viernes, 1 = sábado-domingo)
# ─────────────────────────────────────────────────────────────────────────────
# dayofweek(): Lunes=0, ..., Viernes=4, Sábado=5, Domingo=6
df['es_fin_de_semana'] = (df['fecha'].dt.dayofweek >= 5).astype(int)
print(f"✅ DUMMY 1: es_fin_de_semana")
print(f"   Viernes/Sábado/Domingo: {df['es_fin_de_semana'].sum()} filas")

# ─────────────────────────────────────────────────────────────────────────────
# DUMMY 2: Día Laboral (inverso del anterior)
# ─────────────────────────────────────────────────────────────────────────────
df['es_dia_laboral'] = (df['fecha'].dt.dayofweek < 5).astype(int)
print(f"\n✅ DUMMY 2: es_dia_laboral")
print(f"   Lunes-Viernes: {df['es_dia_laboral'].sum()} filas")

# ─────────────────────────────────────────────────────────────────────────────
# DUMMY 3: Feriados Nacionales Argentina (2021-2025)
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTANTE: Esta es una lista incompleta de ejemplo.
# Para producción, obtener lista completa del calendario oficial argentino

feriados_argentina = [
    # 2021
    '2021-01-01', '2021-02-15', '2021-02-16', '2021-03-24', '2021-04-02',
    '2021-05-01', '2021-05-25', '2021-06-20', '2021-06-21', '2021-07-09',
    '2021-08-17', '2021-10-12', '2021-11-22', '2021-12-08', '2021-12-25',
    # 2022
    '2022-01-01', '2022-02-28', '2022-03-01', '2022-03-24', '2022-04-02',
    '2022-05-01', '2022-05-25', '2022-06-20', '2022-06-21', '2022-07-09',
    '2022-08-17', '2022-10-12', '2022-11-22', '2022-12-08', '2022-12-25',
    # 2023
    '2023-01-01', '2023-02-20', '2023-02-21', '2023-03-24', '2023-04-02',
    '2023-05-01', '2023-05-25', '2023-06-20', '2023-06-21', '2023-07-09',
    '2023-08-17', '2023-10-12', '2023-11-22', '2023-12-08', '2023-12-25',
    # 2024
    '2024-01-01', '2024-02-12', '2024-02-13', '2024-03-24', '2024-04-02',
    '2024-05-01', '2024-05-25', '2024-06-20', '2024-06-21', '2024-07-09',
    '2024-08-17', '2024-10-12', '2024-11-22', '2024-12-08', '2024-12-25',
    # 2025
    '2025-01-01', '2025-03-01', '2025-03-02', '2025-03-24', '2025-04-02',
    '2025-05-01', '2025-05-25', '2025-06-20', '2025-06-21', '2025-07-09',
    '2025-08-17', '2025-10-12', '2025-11-22', '2025-12-08', '2025-12-25'
]

feriados = pd.to_datetime(feriados_argentina)
df['es_feriado'] = df['fecha'].isin(feriados).astype(int)
print(f"\n✅ DUMMY 3: es_feriado")
print(f"   Días feriados: {df['es_feriado'].sum()} filas")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VARIABLES TEMPORALES (para capturar ciclos)
# ─────────────────────────────────────────────────────────────────────────────

# Mes (1-12)
df['mes'] = df['fecha'].dt.month

# Semana del año (1-52)
df['semana_anio'] = df['fecha'].dt.isocalendar().week

# Día del año (1-365/366)
df['dia_anio'] = df['fecha'].dt.dayofyear

# Trimestre (1-4)
df['trimestre'] = df['fecha'].dt.quarter

print(f"\n✅ VARIABLES TEMPORALES")
print(f"   - mes (1-12)")
print(f"   - semana_anio (1-52)")
print(f"   - dia_anio (1-366)")
print(f"   - trimestre (1-4)")

# Verificar estructura final
print(f"\n📊 ESTRUCTURA FINAL DEL DATAFRAME:")
print(f"Columnas ({len(df.columns)}):")
for col in df.columns:
    print(f"  - {col}")

print(f"\nPrimeras filas:")
print(df.head(10))

## Paso 1.6: Validación Final y Guardado del DataFrame

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VALIDACIONES
# ═══════════════════════════════════════════════════════════════════════════════

print("\n✅ VALIDACIONES FINALES\n")

# Validación 1: No hay NaN
nan_count = df.isnull().sum().sum()
if nan_count == 0:
    print(f"✅ Validación 1: NO hay valores NaN")
else:
    print(f"⚠️  Validación 1: Hay {nan_count} valores NaN")
    print(df.isnull().sum())

# Validación 2: No hay duplicados de fecha
duplicados = df['fecha'].duplicated().sum()
if duplicados == 0:
    print(f"✅ Validación 2: NO hay fechas duplicadas")
else:
    print(f"⚠️  Validación 2: Hay {duplicados} fechas duplicadas")

# Validación 3: Fechas están ordenadas
if df['fecha'].is_monotonic_increasing:
    print(f"✅ Validación 3: Fechas están ordenadas correctamente")
else:
    print(f"⚠️  Validación 3: Fechas NO están ordenadas")

# Validación 4: Período está dentro de 2021-2025
fecha_min = df['fecha'].min()
fecha_max = df['fecha'].max()
if fecha_min.year >= 2021 and fecha_max.year <= 2025:
    print(f"✅ Validación 4: Período {fecha_min.date()} a {fecha_max.date()} OK")
else:
    print(f"⚠️  Validación 4: Período fuera de rango")

# Validación 5: Dummies son 0/1
dummy_cols = ['es_fin_de_semana', 'es_dia_laboral', 'es_feriado']
for col in dummy_cols:
    if set(df[col].unique()) <= {0, 1}:
        print(f"✅ Validación 5: {col} contiene solo 0/1")
    else:
        print(f"⚠️  Validación 5: {col} contiene valores diferentes a 0/1")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GUARDAR DATAFRAME EN CSV Y PICKLE
# ═══════════════════════════════════════════════════════════════════════════════

print("\n💾 GUARDANDO DATOS...\n")

# Ruta donde guardaremos (cambiar según tu setup)
ruta_datos = './datos_procesados/'  # Crear esta carpeta localmente

# Crear carpeta si no existe
import os
os.makedirs(ruta_datos, exist_ok=True)

# Guardar CSV (para compartir/ver fácilmente)
csv_path = ruta_datos + 'TP2_datos_preparados.csv'
df.to_csv(csv_path, index=False)
print(f"✅ CSV guardado: {csv_path}")

# Guardar Pickle (para cargar rápido después)
pkl_path = ruta_datos + 'TP2_datos_preparados.pkl'
df.to_pickle(pkl_path)
print(f"✅ Pickle guardado: {pkl_path}")

# Información final
print(f"\n📊 RESUMEN:")
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
print(f"Período: {df['fecha'].min().date()} a {df['fecha'].max().date()}")
print(f"\n💡 PRÓXIMO PASO: Mar y Sarah usarán estos datos para sus modelos")

---

# FASE 2A: MODELO XGBOOST

## Paso 2A.1: Cargar datos preparados y crear features

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cargar el DataFrame que acabamos de guardar
# ALTERNATIVA: Si ya existe en otra carpeta, cargar desde allí
# ═══════════════════════════════════════════════════════════════════════════════

print("\n🔄 CARGANDO DATOS PARA XGBOOST...\n")

# Opción 1: Cargar desde nuestro pickle (más rápido)
df_xgb = pd.read_pickle(pkl_path)
print(f"✅ Datos cargados: {len(df_xgb)} filas, {len(df_xgb.columns)} columnas")

# Opción 2 (ALTERNATIVA): Si los datos están en otra carpeta
# df_xgb = pd.read_csv('C:/ruta/a/datos_guardados/TP2_datos_preparados.csv')
# df_xgb['fecha'] = pd.to_datetime(df_xgb['fecha'])

print(f"\nPrimeras filas:")
print(df_xgb.head())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CREAR FEATURES PARA XGBOOST
# ═══════════════════════════════════════════════════════════════════════════════

print("\n📊 CREANDO FEATURES PARA XGBOOST...\n")

# Seleccionar SERIE A PREDECIR
# En este caso: movilidad urbana
target_col = 'serie_movilidad'
print(f"Serie a predecir: {target_col}")

# ─────────────────────────────────────────────────────────────────────────────
# CREAR LAGS (memoria temporal)
# ─────────────────────────────────────────────────────────────────────────────
# Lag_1: valor de ayer
# Lag_7: valor de hace 7 días (mismo día semana anterior)
# Lag_30: valor de hace 30 días (ciclo mensual)

def crear_lags(data, col, lags=[1, 7, 14, 30]):
    """
    Crea variables de rezago (lag) para capturar dependencia temporal.
    
    Args:
        data: DataFrame
        col: nombre de columna
        lags: lista de lags (días atrás)
    
    Returns:
        data con nuevas columnas: col_lag_1, col_lag_7, etc.
    """
    for lag in lags:
        data[f'{col}_lag_{lag}'] = data[col].shift(lag)
    return data

df_xgb = crear_lags(df_xgb, target_col, lags=[1, 7, 14, 30])
print(f"✅ Lags creados para {target_col}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DEFINIR FEATURES Y TARGET
# ─────────────────────────────────────────────────────────────────────────────

# FEATURES: Variables que el modelo usa para predecir
features = [
    # Lags de la serie principal (memoria temporal)
    'serie_movilidad_lag_1',   # Valor de ayer
    'serie_movilidad_lag_7',   # Mismo día semana anterior
    'serie_movilidad_lag_14',  # Hace 2 semanas
    'serie_movilidad_lag_30',  # Hace 1 mes
    
    # Series exógenas (otras variables que podrían ayudar)
    'serie_congestion',        # Congestión de tráfico
    'serie_temperatura',       # Temperatura
    
    # Variables dummy (patrones semanales/festivos)
    'es_fin_de_semana',
    'es_dia_laboral',
    'es_feriado',
    
    # Variables temporales (ciclos)
    'mes',
    'semana_anio',
    'dia_anio',
    'trimestre'
]

# TARGET: Lo que queremos predecir
# Predecir el valor de MAÑANA usando datos de HOY
df_xgb['target'] = df_xgb[target_col].shift(-1)  # Desplazar -1: valor siguiente

print(f"\n✅ Features definidas: {len(features)}")
for f in features:
    print(f"   - {f}")

print(f"\n✅ Target definido: {target_col} (valor de mañana)")

# Verificar que no hay NaN
print(f"\nVerificando NaN después de crear lags...")
print(df_xgb[features + ['target']].isnull().sum())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ELIMINAR FILAS CON NaN (causadas por lags y target)
# ─────────────────────────────────────────────────────────────────────────────

filas_antes = len(df_xgb)
df_xgb = df_xgb.dropna(subset=features + ['target'])
filas_despues = len(df_xgb)

print(f"\nFiolas ANTES de dropna: {filas_antes}")
print(f"Filas DESPUÉS de dropna: {filas_despues}")
print(f"Filas removidas: {filas_antes - filas_despues}")

print(f"\n✅ Dataset listo para entrenamiento")
print(f"Período: {df_xgb['fecha'].min().date()} a {df_xgb['fecha'].max().date()}")

## Paso 2A.2: Train-Test Split (Respetando cronología)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAIN-TEST SPLIT: Dividir por FECHA (no aleatorio)
# ═══════════════════════════════════════════════════════════════════════════════

# CRÍTICO: En series temporales, NO usamos train_test_split aleatorio
# porque rompe la dependencia temporal.
# En su lugar, dividimos por fecha: primeros años = train, último año = test

print("\n🔀 TRAIN-TEST SPLIT\n")

# Fecha de corte: últimos 365 días para test
fecha_max = df_xgb['fecha'].max()
fecha_corte = fecha_max - timedelta(days=365)

print(f"Fecha de corte: {fecha_corte.date()}")
print(f"Train: hasta {fecha_corte.date()}")
print(f"Test: desde {fecha_corte.date()} + 1 día hasta {fecha_max.date()}")

# Dividir
train_df = df_xgb[df_xgb['fecha'] <= fecha_corte].copy()
test_df = df_xgb[df_xgb['fecha'] > fecha_corte].copy()

print(f"\nTrain: {len(train_df)} filas")
print(f"Test: {len(test_df)} filas")
print(f"Ratio: {100*len(train_df)/(len(train_df)+len(test_df)):.1f}% train, {100*len(test_df)/(len(train_df)+len(test_df)):.1f}% test")

# Extraer X, y
X_train = train_df[features]
y_train = train_df['target']

X_test = test_df[features]
y_test = test_df['target']

print(f"\n✅ Shapes:")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

## Paso 2A.3: Entrenamiento XGBoost

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ENTRENAMIENTO DEL MODELO XGBOOST
# ═══════════════════════════════════════════════════════════════════════════════

print("\n🤖 ENTRENANDO XGBOOST...\n")

# Crear modelo con hiperparámetros
model_xgb = xgb.XGBRegressor(
    # ─ Parámetros de estructura ─
    max_depth=6,               # Profundidad máxima del árbol (evita overfitting)
    learning_rate=0.1,         # Velocidad de aprendizaje (eta)
    n_estimators=300,          # Cantidad de árboles (boosting rounds)
    
    # ─ Parámetros de regularización ─
    subsample=0.8,             # Fracción de datos por árbol (80%)
    colsample_bytree=0.8,      # Fracción de features por árbol (80%)
    min_child_weight=1,        # Peso mínimo en hoja
    gamma=0,                   # Min loss reduction para split
    
    # ─ Otros ─
    random_state=42,           # Para reproducibilidad
    verbosity=0,               # Sin output detallado
    n_jobs=-1                  # Usar todos los CPUs
)

# Entrenar
print("Entrenando modelo...")
model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],  # Para monitorear progreso
    verbose=False
)

print("✅ Modelo entrenado\n")

# Feature importance (qué variables son más importantes)
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model_xgb.feature_importances_
}).sort_values('importance', ascending=False)

print("📊 TOP 10 FEATURES MÁS IMPORTANTES:")
print(feature_importance.head(10))

In [ ]:
# Graficar importancia de features
fig, ax = plt.subplots(figsize=(10, 6))
top_features = feature_importance.head(10)
ax.barh(top_features['feature'], top_features['importance'])
ax.set_xlabel('Importancia')
ax.set_title('XGBoost: Top 10 Features Importantes')
plt.tight_layout()
plt.savefig('02_xgboost_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: 02_xgboost_feature_importance.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GUARDAR MODELO
# ═══════════════════════════════════════════════════════════════════════════════

modelo_path = ruta_datos + 'modelo_xgboost.pkl'
joblib.dump(model_xgb, modelo_path)
print(f"\n💾 Modelo guardado: {modelo_path}")

## Paso 2A.4: Predicciones y Métricas

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PREDICCIONES EN TEST
# ═══════════════════════════════════════════════════════════════════════════════

print("\n🔮 REALIZANDO PREDICCIONES...\n")

y_pred_test = model_xgb.predict(X_test)

# Crear DataFrame con resultados
resultados_test = pd.DataFrame({
    'fecha': test_df['fecha'].values,
    'real': y_test.values,
    'prediccion': y_pred_test
})

print(f"Predicciones en TEST:")
print(resultados_test.head(20))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CÁLCULO DE MÉTRICAS
# ═══════════════════════════════════════════════════════════════════════════════

def calcular_metricas(y_true, y_pred, nombre="XGBoost"):
    """
    Calcula métricas de evaluación para pronóstico de series temporales.
    
    Métricas:
    - SMAPE: Symmetric Mean Absolute Percentage Error (PRINCIPAL)
    - RMSE: Root Mean Squared Error (penaliza outliers)
    - MAE: Mean Absolute Error (promedio de errores absolutos)
    - MAPE: Mean Absolute Percentage Error
    """
    
    # SMAPE: Métrica principal para series temporales
    # Rango: 0-200% (normalmente 0-100%)
    # Mejor: más cercano a 0
    smape = 100 * np.mean(
        2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    )
    
    # RMSE: Error cuadrático medio (penaliza errores grandes)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # MAE: Error absoluto medio (interpretable en unidades originales)
    mae = mean_absolute_error(y_true, y_pred)
    
    # MAPE: Error porcentual absoluto medio
    # Problema: indefinido si y_true=0, evitarlo con máximo
    mape = 100 * np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8)))
    
    return {
        'Modelo': nombre,
        'SMAPE': round(smape, 2),
        'RMSE': round(rmse, 2),
        'MAE': round(mae, 2),
        'MAPE': round(mape, 2)
    }

# Calcular métricas
metricas_xgb = calcular_metricas(y_test.values, y_pred_test, "XGBoost")

print("\n📊 MÉTRICAS DE EVALUACIÓN:")
print("\n" + "="*60)
for key, value in metricas_xgb.items():
    print(f"{key:15s}: {value}")
print("="*60)

# Guardar métricas
metricas_df = pd.DataFrame([metricas_xgb])
metricas_path = ruta_datos + 'metricas_xgboost.csv'
metricas_df.to_csv(metricas_path, index=False)
print(f"\n✅ Métricas guardadas: {metricas_path}")

## Paso 2A.5: Gráficos de Predicción

In [ ]:
# Gráfico 1: Real vs Predicción (TEST)
fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(resultados_test['fecha'], resultados_test['real'], 
        label='Real', color='black', linewidth=2, alpha=0.8)
ax.plot(resultados_test['fecha'], resultados_test['prediccion'], 
        label='Predicción XGBoost', color='blue', linewidth=1.5, alpha=0.7, linestyle='--')

ax.set_xlabel('Fecha', fontsize=11)
ax.set_ylabel('Viajes', fontsize=11)
ax.set_title('XGBoost: Real vs Predicción (TEST 2025)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('03_xgboost_test_vs_real.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: 03_xgboost_test_vs_real.png")

In [ ]:
# Gráfico 2: Error en el tiempo
fig, ax = plt.subplots(figsize=(15, 6))

error = resultados_test['prediccion'] - resultados_test['real']

ax.plot(resultados_test['fecha'], error, color='red', linewidth=1, alpha=0.7)
ax.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax.fill_between(resultados_test['fecha'], error, 0, where=(error >= 0), 
                 color='red', alpha=0.3, label='Sobrestimado')
ax.fill_between(resultados_test['fecha'], error, 0, where=(error < 0), 
                 color='green', alpha=0.3, label='Subestimado')

ax.set_xlabel('Fecha', fontsize=11)
ax.set_ylabel('Error (Predicción - Real)', fontsize=11)
ax.set_title('XGBoost: Error en el Tiempo', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('04_xgboost_error_time.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: 04_xgboost_error_time.png")

## Paso 2A.6: Pronóstico para 2026

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PRONÓSTICO PARA 2026 (365 días)
# ═══════════════════════════════════════════════════════════════════════════════

print("\n🔮 GENERANDO PRONÓSTICO 2026...\n")

# Crear fechas para 2026
fechas_2026 = pd.date_range('2026-01-01', '2026-12-31', freq='D')

# Usar últimas filas de datos para extraer lags iniciales
last_row = df_xgb.iloc[-1].copy()

pronosticos_2026 = []

for i, fecha in enumerate(fechas_2026):
    # Crear fila para predicción
    row_pred = pd.DataFrame([last_row[features]]).copy()
    
    # Hacer predicción
    pred = model_xgb.predict(row_pred)[0]
    
    pronosticos_2026.append({
        'fecha': fecha,
        'prediccion': pred
    })
    
    # Actualizar last_row para siguiente iteración
    # (usar predicción como nuevo lag_1)
    last_row['serie_movilidad_lag_1'] = last_row['serie_movilidad_lag_7']
    last_row['serie_movilidad_lag_7'] = last_row['serie_movilidad_lag_14']
    last_row['serie_movilidad_lag_14'] = last_row['serie_movilidad_lag_30']
    last_row['serie_movilidad_lag_30'] = pred
    
    # Actualizar variables temporales
    last_row['fecha'] = fecha
    last_row['mes'] = fecha.month
    last_row['semana_anio'] = fecha.isocalendar()[1]
    last_row['dia_anio'] = fecha.dayofyear
    last_row['trimestre'] = fecha.quarter
    last_row['es_fin_de_semana'] = 1 if fecha.dayofweek >= 5 else 0
    last_row['es_dia_laboral'] = 1 if fecha.dayofweek < 5 else 0
    last_row['es_feriado'] = 1 if fecha in feriados else 0
    
    if (i + 1) % 100 == 0:
        print(f"Generadas {i + 1}/365 predicciones...")

pronostico_2026_df = pd.DataFrame(pronosticos_2026)
print(f"\n✅ Pronóstico 2026 generado: {len(pronostico_2026_df)} días")
print(f"\nPrimeras predicciones:")
print(pronostico_2026_df.head(20))

In [ ]:
# Guardar pronóstico 2026
pronostico_path = ruta_datos + 'predicciones_xgboost_2026.csv'
pronostico_2026_df.to_csv(pronostico_path, index=False)
print(f"✅ Pronóstico guardado: {pronostico_path}")

In [ ]:
# Gráfico 3: Pronóstico 2026
fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(pronostico_2026_df['fecha'], pronostico_2026_df['prediccion'], 
        color='green', linewidth=1.5, alpha=0.8)

ax.set_xlabel('Fecha', fontsize=11)
ax.set_ylabel('Viajes (Pronóstico)', fontsize=11)
ax.set_title('XGBoost: Pronóstico para 2026', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('05_xgboost_pronostico_2026.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: 05_xgboost_pronostico_2026.png")

# Resumen Final

In [ ]:
print("\n" + "="*80)
print("RESUMEN FINAL - TP2 FASE 1 + FASE 2A (AGUS)")
print("="*80)

print("\n✅ FASE 1 COMPLETADA:")
print(f"   • 3 datasets unificados en 1 DataFrame")
print(f"   • {len(outliers_log)} outliers removidos y documentados")
print(f"   • 8 variables dummy/temporales creadas")
print(f"   • {len(df)} registros finales (2021-2025)")

print(f"\n✅ FASE 2A COMPLETADA:")
print(f"   • Modelo XGBoost entrenado")
print(f"   • {len(y_test)} predicciones en TEST")
print(f"   • Métricas SMAPE={metricas_xgb['SMAPE']}%, RMSE={metricas_xgb['RMSE']}")
print(f"   • 365 predicciones para 2026")

print(f"\n💾 ARCHIVOS GUARDADOS:")
print(f"   • {csv_path}")
print(f"   • {pkl_path}")
print(f"   • {modelo_path}")
print(f"   • {pronostico_path}")
print(f"   • {metricas_path}")
print(f"   • outliers_removidos.csv")
print(f"   • Gráficos: 01_... a 05_...")

print(f"\n👥 PRÓXIMOS PASOS:")
print(f"   1. MAR: Usar CSV para entrenar Prophet")
print(f"   2. SARAH: Usar CSV para entrenar LSTM")
print(f"   3. GIULI: Comparar predicciones de 3 modelos")

print("\n" + "="*80)
print("¡AGUS TERMINA AQUÍ. Mar y Sarah pueden comenzar!")
print("="*80)